In [2]:
import pandas as pd
import re
from urllib.parse import unquote
import ast

df = pd.read_json("urls.jsonl.gz", lines=True, nrows=3000000)
df

,display_id,urls
0,0o2M2ZebIqs,[http://www.youtube.com/watch?v=lepXEdv1Udc]
1,VPqJmODeZyk,"[https://goo.gl/JI78Ak, http://www.youtube.com..."
2,xLYGF-aCEHk,"[https://goo.gl/JI78Ak, http://www.youtube.com..."
3,TqzfdwSZdRc,"[https://goo.gl/JI78Ak, http://www.youtube.com..."
4,C-dn3p-ZTrM,"[https://goo.gl/JI78Ak, http://www.youtube.com..."
...,...,...
2999995,49FjXJ7h5jw,"[http://bit.ly/Um9HH4, http://bit.ly/Yd5TtU, h..."
2999996,S48-2cl2VjU,"[http://bit.ly/Um9HH4, http://bit.ly/Yd5TtU, h..."
2999997,1S_aqUDb62Q,"[http://bit.ly/Um9HH4, http://bit.ly/Yd5TtU, h..."
2999998,Afwl3C0tPhY,"[http://FB.com/FunkerTactical, http://Machinim..."


In [5]:
edu_domains = [
    r'wikipedia\.org',
    r'\.edu\b',
    r'\.ac\.[a-z]{2}\b',
    r'\.gov\b',
    r'researchgate\.net',
    r'arxiv\.org',
    r'nature\.com',
    r'sciencedirect\.com',
    r'springer\.com',
    r'elsevier\.com',
    r'jstor\.org',
    r'cambridge\.org',
    r'oxfordjournals\.org',
    r'pnas\.org',
    r'nih\.gov',
    r'nasa\.gov',
    r'plos\.org',
    r'biorxiv\.org',
    r'medrxiv\.org'
]

# Compile regex
edu_pattern = re.compile('|'.join(edu_domains), re.IGNORECASE)

# Extract matching domains
df_new = pd.DataFrame({
    'academic_sites': df['urls'].apply(
        lambda links: list({match.group(0).lower() for url in links if (match := edu_pattern.search(url))})
    )
})

In [26]:
main_sites = [
    'wikipedia.org',
    'researchgate.net',
    'arxiv.org',
    'nature.com',
    'sciencedirect.com',
    'springer.com',
    'elsevier.com',
    'jstor.org',
    'cambridge.org',
    'oxfordjournals.org',
    'pnas.org',
    'nih.gov',
    'nasa.gov',
    'plos.org',
    'biorxiv.org',
    'medrxiv.org',
    'britannica.com',
    'aps.org',
    'acm.org',
    'ieee.org',
    'un.org',
    'who.int',
    'sciencenews.org',
    'newscientist.com',
    'mit.edu',
    'stanford.edu',
    'harvard.edu',
    'ed.gov',
    'khanacademy.org',
    'coursera.org',
    'edx.org'
]

from urllib.parse import urlparse
def extract_main_sites(urls):
    domains = set()
    for url in urls:
        domain = urlparse(url).netloc.lower().replace('www.', '')
        for site in main_sites:
            if site in domain:
                domains.add(site)
                break  # stop checking other sites once matched
    return list(domains)

df_new = pd.DataFrame({
    'academic_sites': df['urls'].apply(extract_main_sites)
}).explode('academic_sites').dropna().reset_index(drop=True)



In [31]:
import numpy as np
df_new

site_counts = df_new['academic_sites'].value_counts()

site_counts_df = site_counts.reset_index()
site_counts_df.columns = ['website', 'count']
site_counts_df['log_count'] = np.log(site_counts_df['count'])  # natural log
site_counts_df


,website,count,log_count
0,wikipedia.org,5331,8.581294
1,nasa.gov,1183,7.075809
2,nature.com,913,6.816736
3,nih.gov,757,6.629363
4,harvard.edu,388,5.961005
5,sciencedirect.com,232,5.446737
6,arxiv.org,205,5.323010
7,newscientist.com,177,5.176150
8,mit.edu,150,5.010635
9,pnas.org,134,4.897840


In [33]:
import plotly.express as px
import pandas as pd

# Assuming `site_counts_df` was created like this:
# site_counts_df = df_new['academic_sites'].value_counts().reset_index()
# site_counts_df.columns = ['website', 'count']

top_n = 29
site_counts_top = site_counts_df.head(top_n)

# Create interactive pie chart
fig = px.pie(
    site_counts_top,
    values='log_count',
    names='website',
    #title='Top 20 Academic Websites',
    hole=0.3,  # donut style
)

# Style adjustments
fig.update_traces(
    textposition='outside',
    textinfo='label+value',
    pull=[0.03]*len(site_counts_top),
)

# Clean layout
fig.update_layout(
    title_font=dict(size=22, family='Arial', color='black'),
    showlegend=False,
    margin=dict(t=60, b=40, l=40, r=40)
)

# Save and show
fig.write_html("edu_sites_pie.html")